# AraT5v2 - experiments to explain the gap with the paper

Our first run: BLEU 76.21. Paper: 79.78. These runs change **one thing at a time** to find out why.

| Run | What changes | Question answered |
|---|---|---|
| A | keep the best (step 300) checkpoint | does the save/eval step mismatch matter? |
| B | authors' protocol: 2550 train, validation = test[:80] | is the gap due to their setup? |
| C (optional) | seeds 1 and 2 | how much does luck matter? |

**Before you start:** Settings -> Accelerator = *GPU T4 x2*, *internet ON*. You must have pushed the new code to GitHub
(the flags `--save_steps`, `--splits_dir`, `--tag` only exist in the new version).
Run cells **one at a time, top to bottom**. Red dependency-conflict warnings are harmless.

## 1. Get the code (once per fresh session)

In [ ]:
import os, glob, subprocess

# >>> EDIT THIS LINE: your GitHub username <<<
REPO_URL = "https://github.com/YOUR_USERNAME/Reproduction-of-domain-specific-Tunisian-Arabic-dialect-to-MSA-translation.git"

WORK = "/kaggle/working"
name = REPO_URL.rstrip("/").removesuffix(".git").split("/")[-1]
path = os.path.join(WORK, name)
if not os.path.exists(path):
    subprocess.run(["git", "clone", REPO_URL, path], check=True)
else:
    subprocess.run(["git", "-C", path, "checkout", "--", "."], check=True)
    subprocess.run(["git", "-C", path, "pull"], check=True)

hits = sorted(glob.glob(path + "/**/src/train_seq2seq.py", recursive=True), key=len)
assert hits, "src/train_seq2seq.py not found: check REPO_URL and that you pushed the files"
os.chdir(os.path.dirname(os.path.dirname(hits[0])))
print("working directory:", os.getcwd())
src = open("src/train_seq2seq.py", encoding="utf-8").read()
print("NEW code present (has --save_steps flag):", "--save_steps" in src)
assert "--save_steps" in src, "Old code on GitHub: push the new files first (git add . / commit / push), then re-run this cell"

## 2. Install the right library versions (once per fresh session)

In [ ]:
!pip install -q "transformers>=4.46,<5" sacrebleu pyyaml sentencepiece protobuf

## 3. Check the environment: must say transformers 4.x and GPUs visible 1 or 2

In [ ]:
!python -c "import transformers, torch; print('transformers', transformers.__version__, '| GPU available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())"

## 4. Build the data splits and run the safety tests

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.data
!python -m src.data --protocol authors
!python -m tests.test_against_stored
!python -m tests.test_protocols

You must see `ALL CHECKS PASSED` and `test_protocols: OK`. If not, stop and send me the output.

## 5. Run A - keep the best checkpoint (about 6 min)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.train_seq2seq --config configs/seq2seq_arat5v2.yaml --save_steps 100 --tag save100

## 6. Run B - the authors' protocol (about 6 min)

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.train_seq2seq --config configs/seq2seq_arat5v2.yaml --splits_dir data/splits_authors --tag authors

## 7. SUMMARY TABLE - send me this output as text

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

import json
print(f"{'run':<56}{'BLEU':>7}{'chrF++':>8}{'TER':>7}{'EM':>7}  checkpoint")
for f in sorted(glob.glob("results/table7_seq2seq/*/metrics.json")):
    r = json.load(open(f, encoding="utf-8")); m, t = r["metrics"], r["meta"]
    print(f"{t['run']:<56}{m['bleu_char']:>7}{m['chrf_pp']:>8}{m['ter']:>7}{m['exact_match']:>7}  {os.path.basename(t['best_checkpoint'] or '-')}")
print("PAPER (authors' file)                                    79.78  74.96  25.32  31.78")

## 8. OPTIONAL Run C - seed variance (about 12 min). Only if you have time.

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

!python -m src.train_seq2seq --config configs/seq2seq_arat5v2.yaml --save_steps 100 --seed 1 --tag save100
!python -m src.train_seq2seq --config configs/seq2seq_arat5v2.yaml --save_steps 100 --seed 2 --tag save100

## 9. Run the SUMMARY cell (7) again if you ran C, then package the results

In [ ]:
import os, glob
_hits = sorted(glob.glob("/kaggle/working/**/src/train_seq2seq.py", recursive=True), key=len)
assert _hits, "Repo not found under /kaggle/working: run cell 1 first"
os.chdir(os.path.dirname(os.path.dirname(_hits[0])))
print("running in:", os.getcwd())

import shutil
!zip -qr results_arat5v2_experiments.zip results
shutil.copy("results_arat5v2_experiments.zip", "/kaggle/working/results_arat5v2_experiments.zip")
!ls -la /kaggle/working/results_arat5v2_experiments.zip

The zip is now directly under `/kaggle/working` (Output panel on the right -> three dots -> Download). Sending me the cell 7 table is enough; the zip is optional.